# miso Notebook: Organize `messages` and build a Python-writing agent

This notebook shows two things:
1. How to structure a clean `messages` list for `agent.run(...)`
2. How to make an agent that writes Python files for the user via tool calls


## 1) Message design rules

Use this order:
- `system`: stable behavior and constraints
- `user` (context): repo/project background
- `user` (task): exact deliverable and file path
- optional follow-up `user`: extra constraints

Keep each message focused. Do not mix policy, context, and task in one long block.

In [1]:
from typing import Dict, List

SYSTEM_PROMPT = """You are a software engineer agent.
When the user asks for a Python file:
- Use tools when available.
- If write_text_file is available, create/update the requested file.
- Return a short summary after tool execution.
"""

def build_messages(task: str, *, context: str = "", constraints: str = "") -> List[Dict[str, str]]:
    messages: List[Dict[str, str]] = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
    ]

    if context.strip():
        messages.append({
            "role": "user",
            "content": f"Project context:\n{context.strip()}"
        })

    messages.append({"role": "user", "content": task.strip()})

    if constraints.strip():
        messages.append({
            "role": "user",
            "content": f"Constraints:\n{constraints.strip()}"
        })

    return messages

def show_messages(messages: List[Dict[str, str]]) -> None:
    for idx, msg in enumerate(messages, start=1):
        role = msg.get("role", "?")
        content = msg.get("content", "")
        print(f"[{idx}] {role}\n{content}\n")


In [2]:
task = """
Create a Python file named user.py.
Requirements:
- Define a dataclass User with fields: id (int), name (str), email (str)
- Add method to_dict(self) -> dict
- Add a __main__ block that creates one sample user and prints JSON
- Use write_text_file tool to create the file
"""

messages = build_messages(
    task,
    context="This repository is a lightweight agent builder demo.",
    constraints="Use only Python standard library; keep code short and readable."
)

show_messages(messages)


[1] system
You are a software engineer agent.
When the user asks for a Python file:
- Use tools when available.
- If write_text_file is available, create/update the requested file.
- Return a short summary after tool execution.

[2] user
Project context:
This repository is a lightweight agent builder demo.

[3] user
Create a Python file named user.py.
Requirements:
- Define a dataclass User with fields: id (int), name (str), email (str)
- Add method to_dict(self) -> dict
- Add a __main__ block that creates one sample user and prints JSON
- Use write_text_file tool to create the file

[4] user
Constraints:
Use only Python standard library; keep code short and readable.



## 2) Build the agent

This uses the predefined toolkit, so the model can call `write_text_file`.

Provider selection:
- `MISO_PROVIDER=openai` with `OPENAI_API_KEY`
- `MISO_PROVIDER=ollama` with local Ollama server


In [ ]:
import os
from miso import agent as Agent

agent = Agent()
provider = os.getenv("MISO_PROVIDER", "ollama").lower()

if provider == "openai":
    agent.provider = "openai"
    agent.model = os.getenv("OPENAI_MODEL", "gpt-4.1")
    agent.openai_api_key = os.getenv("OPENAI_API_KEY")
    agent.openai_base_url = os.getenv("OPENAI_BASE_URL") or None
    if not agent.openai_api_key:
        raise ValueError("OPENAI_API_KEY is required when MISO_PROVIDER=openai")
elif provider == "ollama":
    agent.provider = "ollama"
    agent.model = os.getenv("OLLAMA_MODEL", "deepseek-r1:14b")
else:
    raise ValueError("MISO_PROVIDER must be either 'openai' or 'ollama'")

agent.use_predefined_toolkit(workspace_root=".", include_python_runtime=False)
print(f"provider={agent.provider}, model={agent.model}")


In [ ]:
events = []

def on_event(evt: dict) -> None:
    events.append(evt)
    if evt.get("type") in {"tool_call", "tool_result", "final_message"}:
        print("event:", evt.get("type"), "tool:", evt.get("tool_name", "-"))

payload = {"max_output_tokens": 1200} if agent.provider == "openai" else {"num_predict": 1200}

result = agent.run(
    messages=messages,
    payload=payload,
    callback=on_event,
    max_iterations=4,
)

last_assistant = [m for m in result if isinstance(m, dict) and m.get("role") == "assistant"][-1]
print("\nAssistant final message:\n", last_assistant.get("content", ""))


In [ ]:
from pathlib import Path

generated = Path("user.py")
if generated.exists():
    print("Generated:", generated.resolve())
    print("\n--- user.py ---\n")
    print(generated.read_text(encoding="utf-8"))
else:
    print("user.py not found. Check model output and tool-call events.")


## 3) Reusable helper: generate any Python file for the user


In [ ]:
def generate_python_file_for_user(user_request: str, output_path: str = "user.py"):
    task = f"""
Please create a Python file at {output_path}.
User request:
{user_request}

Use write_text_file tool to write the file.
"""

    msgs = build_messages(
        task,
        context="This repo uses miso agent with predefined toolkit.",
        constraints="Use clear function names and include short docstrings."
    )

    p = {"max_output_tokens": 1200} if agent.provider == "openai" else {"num_predict": 1200}
    return agent.run(messages=msgs, payload=p, max_iterations=4)

# Example:
# generate_python_file_for_user(
#     user_request="Create a small CLI todo manager with add/list commands.",
#     output_path="todo_cli.py",
# )
